# 001 OpenAI Chat Basics

这是第一份 OpenAI 学习 Notebook。

学习目标：

1. 学会在 Notebook 中加载项目 `.env`
2. 学会创建 OpenAI Python Client
3. 学会区分 OpenAI 官方 `Responses API` 和私有兼容网关的 `Chat Completions API`
4. 跑通最简单的单轮聊天功能

你当前使用的是私有模型网关：`OPENAI_BASE_URL=http://192.168.102.19:8082/v1`，模型是 `qwq`。这个网关支持 `/v1/chat/completions`，但不支持 `/v1/responses`，所以本 Notebook 会自动切换到兼容模式。


## 使用前准备

请先确保：

1. 已安装 `openai` 和 `python-dotenv`
2. 项目根目录 `.env` 中已配置 `OPENAI_API_KEY`
3. 当前 Notebook Kernel 选择的是项目虚拟环境 `.venv`

如果还没装依赖，可以先运行下一格。


In [30]:
%pip install -U openai python-dotenv


5995.56s - pydevd: Sending message related to process being replaced timed-out after 5 seconds


/home/dev/bxc/fastapi-study/.venv/bin/python: No module named pip
Note: you may need to restart the kernel to use updated packages.


## 加载环境变量

这里会自动向上查找项目根目录中的 `.env` 文件。

你的私有模型配置可以类似这样：

```env
OPENAI_API_KEY=任意非空值或你的网关密钥
OPENAI_MODEL=qwq
OPENAI_BASE_URL=http://192.168.102.19:8082/v1
```

如果你使用 OpenAI 官方接口，则通常这样配置：

```env
OPENAI_API_KEY=your-openai-api-key
OPENAI_MODEL=gpt-5.4-mini
OPENAI_BASE_URL=
```


In [31]:
import os
from pathlib import Path

from dotenv import load_dotenv


def load_project_env() -> Path | None:
    current = Path.cwd().resolve()
    for path in [current, *current.parents]:
        env_path = path / ".env"
        if env_path.exists():
            load_dotenv(env_path, override=False)
            return env_path
    return None


env_path = load_project_env()
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
OPENAI_MODEL = os.getenv("OPENAI_MODEL", "gpt-5.4-mini")
OPENAI_BASE_URL = os.getenv("OPENAI_BASE_URL") or None

print(f"Loaded .env: {env_path}" if env_path else "No .env found")
print("OPENAI_MODEL =", OPENAI_MODEL)
print("OPENAI_API_KEY loaded =", bool(OPENAI_API_KEY))
print("OPENAI_BASE_URL =", OPENAI_BASE_URL)


Loaded .env: /home/dev/bxc/fastapi-study/.env
OPENAI_MODEL = qwq
OPENAI_API_KEY loaded = True
OPENAI_BASE_URL = http://192.168.102.19:8082/v1


## 探测当前接口模式

OpenAI 官方接口推荐优先学习 `Responses API`。

但很多私有模型网关虽然“遵循 OpenAI 范式”，实际只兼容老一些的 `/v1/chat/completions`。

本项目已经用 `curl` 验证过你的网关：

- `GET /v1/models` 正常
- `POST /v1/chat/completions` 正常
- `POST /v1/responses` 返回 `Internal Server Error`

所以当前 `qwq` 私有模型应该走 `chat.completions.create(...)`。


In [32]:
def is_private_compatible_gateway(base_url: str | None) -> bool:
    if not base_url:
        return False
    return "api.openai.com" not in base_url


API_MODE = "chat_completions" if is_private_compatible_gateway(OPENAI_BASE_URL) else "responses"
print("API_MODE =", API_MODE)


API_MODE = chat_completions


## 创建 OpenAI Client

`OpenAI` 这个 SDK 客户端既可以访问 OpenAI 官方接口，也可以访问兼容 OpenAI 协议的私有网关。

区别主要在调用方法：

- 官方 Responses API：`client.responses.create(...)`
- 兼容网关常见写法：`client.chat.completions.create(...)`


In [33]:
from openai import OpenAI


if not OPENAI_API_KEY:
    raise ValueError("请先在项目根目录 .env 中配置 OPENAI_API_KEY")


client = OpenAI(
    api_key=OPENAI_API_KEY,
    base_url=OPENAI_BASE_URL,
)


SYSTEM_PROMPT = """
你是一个教学型聊天助手。
回答时请：
1. 使用中文
2. 解释清楚概念
3. 尽量简洁
""".strip()


## 封装聊天函数

这个 `chat_once(...)` 会根据 `API_MODE` 自动选择调用方式。

你的私有 `qwq` 网关会走 `chat_completions` 分支。


In [34]:
def chat_once_with_responses_api(
    message: str,
    system_prompt: str = SYSTEM_PROMPT,
    model: str = OPENAI_MODEL,
) -> str:
    response = client.responses.create(
        model=model,
        instructions=system_prompt,
        input=message,
    )
    return response.output_text


def chat_once_with_chat_completions(
    message: str,
    system_prompt: str = SYSTEM_PROMPT,
    model: str = OPENAI_MODEL,
) -> str:
    response = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": message},
        ],
    )
    return response.choices[0].message.content or ""


def chat_once(message: str, system_prompt: str = SYSTEM_PROMPT, model: str = OPENAI_MODEL) -> str:
    if API_MODE == "responses":
        return chat_once_with_responses_api(message, system_prompt=system_prompt, model=model)
    return chat_once_with_chat_completions(message, system_prompt=system_prompt, model=model)


## 先试一个最简单的问题

这格现在会调用 `chat_once(...)`。在你的私有网关配置下，它会实际请求 `/v1/chat/completions`。


In [36]:
reply = chat_once("你好，请用 3 句话介绍什么是 FastAPI")
print(reply)


Value(False)


FastAPI 是一个基于 Python 的现代高性能 Web 框架，专为快速构建 RESTful API 而生。它深度结合 Python 类型提示，可自动完成请求数据校验、生成交互式 API 文档并原生支持异步编程。凭借极快的运行速度和优异的开发体验，它已成为当前 Python 后端开发的主流选择之一。


## 自己提问

把下面的 `my_question` 改成你自己的问题，再运行。


In [37]:
my_question = "我是 Java 开发者，想学习 Python 异步，请给我一个入门建议"
reply = chat_once(my_question)
print(reply)




作为 Java 开发者，学习 Python 异步需先完成**思维模型切换**。以下是精简入门建议：

### 🔑 一、核心思维转变
| Java 异步模型 | Python `asyncio` 模型 |
|---|---|
| 多线程 + 抢占式调度 | 单线程事件循环 + 协作式调度 |
| `CompletableFuture` / `Future` | `async def` 协程函数 |
| 阻塞 I/O 会占线程 | **任何阻塞操作都会卡死整个事件循环** |
| CPU 密集可用线程池并行 | 专为 **I/O 密集** 设计，CPU 密集建议走线程/多进程 |

### 📖 二、语法与 API 对照
- `CompletableFuture.supplyAsync()` → `async def func(): ...`
- `future.get()` → `await coroutine`
- `ExecutorService.submit()` → 直接 `await`（由事件循环调度，无需手动建线程池）
- 并发控制：`asyncio.gather()` / `asyncio.as_completed()` / `asyncio.Semaphore`
- 关键：所有 I/O 必须用异步库（如 `requests` → `httpx`，`pymysql` → `aiomysql`）

### 🛠 三、推荐学习路径
1. **基础语法**：掌握 `async`/`await`、协程对象、`asyncio.run()`
2. **事件循环**：理解 `loop.run_until_complete()` 及任务调度机制
3. **并发实践**：用 `gather`/`as_completed` 实现高并发 I/O
4. **框架落地**：用 `FastAPI` 或 `aiohttp` 写异步 Web 服务（天然支持）
5. **混合场景**：学习 `loop.run_in_executor` 安全调用同步代码

### ⚠️ 四、常见陷阱
- ❌ 在 `async` 函数中调用同步阻塞代码（如 `time.sleep`、同步 HTTP 请求）→ 全局卡死
- ❌ 用 `threading` 替代 `asyncio` 解决 I/O 瓶颈 → 失去

## 查看原始返回结构

学习阶段建议你看一次原始返回。

你的 `qwq` 网关返回里除了 `message.content`，还可能包含 `reasoning_content`。真正给用户展示时，一般只取 `content`。


In [ ]:
raw_response = client.chat.completions.create(
    model=OPENAI_MODEL,
    messages=[
        {"role": "user", "content": "请用一句话介绍 FastAPI"},
    ],
)

print("content:")
print(raw_response.choices[0].message.content)

print("reasoning_content exists:")
print(hasattr(raw_response.choices[0].message, "reasoning_content"))


## 当前结论

你这类私有模型网关虽然使用 OpenAI SDK，但学习时要区分两层含义：

1. OpenAI SDK：Python 客户端库，可以连接官方接口，也可以连接兼容网关
2. OpenAI Responses API：官方较新的接口，不是所有兼容网关都支持

所以当前阶段：

- 使用官方 OpenAI：优先学 `responses.create(...)`
- 使用你的私有 `qwq` 网关：先用 `chat.completions.create(...)`

后续学习多轮对话、结构化输出、函数调用时，也要先确认私有网关支持到什么程度。


## 下一步建议

当这一份 Notebook 跑通后，下一份建议学习：

- `002-openai-chat-history.ipynb`

到时会进入：

- 多轮对话
- 自己维护 messages 历史
- 兼容私有网关的上下文传递方式
